In [ ]:
import pandas as pd

output_filename = "CIRA-CIC-DoHBrw-2020.csv"
df_final = pd.read_csv(output_filename)

# ------------------------------------------------------------
# 7. Final message
# ------------------------------------------------------------
print("File generated successfully!")
print(f"Total number of rows: {df_final.shape[0]}")
print(f"Total number of columns: {df_final.shape[1]}")




# Remove strong identifiers
# List of columns you want to remove
columns_to_remove = ['SourceIP', 'DestinationIP', 'SourcePort', 'DestinationPort', 'TimeStamp']



# Selects all columns except the listed ones
df_filtrado = df_final.drop(columns=columns_to_remove)

# Displays the resulting DataFrame
print(df_filtrado)


# --- MAPPING FOR MALICIOUS DETECTION (CLASS 1) ---
mapping = {
    'DoH': 0,
    'NonDoH': 0,
    'Benign': 0,
    'Malicious': 1
}

# Applying the mapping
df_filtrado.iloc[:, -1] = df_filtrado.iloc[:, -1].map(mapping)

# Removing possible values that were not included in the mapping and ensuring int32
df_filtrado.dropna(subset=[df_filtrado.columns[-1]], inplace=True)
#y = df.iloc[:, -1].astype('int32').copy()

print("Class distribution (0 = Normal / 1 = Malicious):")
#print(y.value_counts())

In [ ]:
%%writefile ids_engine.py
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import multiprocessing as mp
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE

# ------------------------------------------------------------
# MEMORY PROTECTIONS
# ------------------------------------------------------------
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ------------------------------------------------------------
# GENERAL SETTINGS
# ------------------------------------------------------------
BATCH_SIZE = 32
ADV_BATCH_SIZE = 64
EPOCHS = 100
EPSILONS = [0.001, 0.005, 0.01, 0.02, 0.05]
N_RUNS = 30

# ------------------------------------------------------------
# CIRA PLAUSIBILITY MASK
# ------------------------------------------------------------
# The CIRA dataframe used here has the following expected structure:
# - identifier / metadata columns:
#   SourceIP, DestinationIP, SourcePort, DestinationPort, TimeStamp
# - 29 numerical traffic features;
# - one target column: Label or DoH.
#
# Label/DoH is a response variable and must never be perturbed.
#
# Mask convention:
# - 1.0 means the feature can be perturbed;
# - 0.0 means the feature is blocked.
#
# Strategy:
# - deny-by-default: every feature is blocked unless explicitly allowed;
# - the default mask is OPERATIONAL, i.e., it allows timing, packet-length and
#   flow-volume/rate features that are plausible under traffic pacing, packet
#   size changes, padding/fragmentation and controlled request patterns.
#
# Scientific rationale:
# - IPs, ports, timestamps and labels are not continuous adversarial variables;
# - derived variables must be interpreted as the result of coherent changes in
#   the underlying traffic, not as arbitrary independent edits.

CIRA_EXPECTED_FEATURES = 29

CIRA_IDENTIFIER_COLUMNS = [
    "SourceIP",
    "DestinationIP",
    "SourcePort",
    "DestinationPort",
    "TimeStamp",
]

CIRA_TARGET_COLUMNS = ["Label", "DoH"]

# Choose one of: "strict", "operational", "extended".
# strict      -> only Duration + PacketTime* features;
# operational -> Duration + Flow* + PacketLength* + PacketTime*;
# extended    -> operational + ResponseTimeTime* features.
CIRA_MASK_MODE = "operational"

CIRA_ALLOWED_FEATURES_STRICT = [
    "Duration",
    "PacketTimeVariance",
    "PacketTimeStandardDeviation",
    "PacketTimeMean",
    "PacketTimeMedian",
    "PacketTimeMode",
    "PacketTimeSkewFromMedian",
    "PacketTimeSkewFromMode",
    "PacketTimeCoefficientofVariation",
]

CIRA_ALLOWED_FEATURES_OPERATIONAL = [
    "Duration",

    "FlowBytesSent",
    "FlowSentRate",
    "FlowBytesReceived",
    "FlowReceivedRate",

    "PacketLengthVariance",
    "PacketLengthStandardDeviation",
    "PacketLengthMean",
    "PacketLengthMedian",
    "PacketLengthMode",
    "PacketLengthSkewFromMedian",
    "PacketLengthSkewFromMode",
    "PacketLengthCoefficientofVariation",

    "PacketTimeVariance",
    "PacketTimeStandardDeviation",
    "PacketTimeMean",
    "PacketTimeMedian",
    "PacketTimeMode",
    "PacketTimeSkewFromMedian",
    "PacketTimeSkewFromMode",
    "PacketTimeCoefficientofVariation",
]

CIRA_ALLOWED_FEATURES_EXTENDED = [
    *CIRA_ALLOWED_FEATURES_OPERATIONAL,

    "ResponseTimeTimeVariance",
    "ResponseTimeTimeStandardDeviation",
    "ResponseTimeTimeMean",
    "ResponseTimeTimeMedian",
    "ResponseTimeTimeMode",
    "ResponseTimeTimeSkewFromMedian",
    "ResponseTimeTimeSkewFromMode",
    "ResponseTimeTimeCoefficientofVariation",
]


def get_cira_allowed_features(mask_mode=CIRA_MASK_MODE):
    """
    Returns the CIRA perturbable feature list for the selected plausibility mask.

    Parameters
    ----------
    mask_mode : str
        One of {"strict", "operational", "extended"}.

    Returns
    -------
    list[str]
        Feature names allowed to receive adversarial perturbations.
    """
    mode = str(mask_mode).strip().lower()

    if mode == "strict":
        return list(CIRA_ALLOWED_FEATURES_STRICT)

    if mode == "operational":
        return list(CIRA_ALLOWED_FEATURES_OPERATIONAL)

    if mode == "extended":
        return list(CIRA_ALLOWED_FEATURES_EXTENDED)

    raise ValueError(
        "Invalid CIRA mask mode. Expected 'strict', 'operational', or "
        f"'extended', but received: {mask_mode!r}"
    )


CIRA_ALLOWED_FEATURES = get_cira_allowed_features(CIRA_MASK_MODE)


def encode_cira_target(y):
    """
    Encodes the CIRA target column as int32.

    Supported values include:
    - boolean True/False;
    - numeric 1/0;
    - string variants such as DoH/NonDoH, True/False, Malicious/Benign,
      Attack/Normal.
    """
    y_series = pd.Series(y).copy()

    if pd.api.types.is_bool_dtype(y_series):
        return y_series.astype("int32")

    if pd.api.types.is_numeric_dtype(y_series):
        return y_series.astype("int32")

    normalized = (
        y_series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace("-", "", regex=False)
        .str.replace("_", "", regex=False)
        .str.replace(" ", "", regex=False)
    )

    mapping = {
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0,
        "doh": 1,
        "nondoh": 0,
        "non-doh": 0,
        "nondns": 0,
        "malicious": 1,
        "benign": 0,
        "attack": 1,
        "normal": 0,
        "anomaly": 1,
        "legitimate": 0,
    }

    encoded = normalized.map(mapping)

    if encoded.isna().any():
        unknown_values = sorted(y_series[encoded.isna()].astype(str).unique().tolist())
        raise ValueError(
            "Unknown CIRA target values found in the target column: "
            f"{unknown_values}. Extend encode_cira_target() with the dataset labels."
        )

    return encoded.astype("int32")


def select_cira_features_and_target(df):
    """
    Selects CIRA model features and binary target.

    Supported inputs:
    1. Full CIRA dataframe:
       SourceIP, DestinationIP, SourcePort, DestinationPort, TimeStamp,
       29 traffic features, and Label/DoH.
    2. Filtered CIRA dataframe:
       29 traffic features and Label/DoH.
    3. Generic dataframe:
       29 traffic features and target as the last column.

    Label/DoH is a response variable and must never be included in X or in the
    adversarial plausibility mask.
    """
    df = df.copy()

    target_col = None
    for candidate in CIRA_TARGET_COLUMNS:
        if candidate in df.columns:
            target_col = candidate
            break

    if target_col is not None:
        y = encode_cira_target(df[target_col])
        feature_df = df.drop(columns=[target_col])
    else:
        y = encode_cira_target(df.iloc[:, -1])
        feature_df = df.iloc[:, :-1].copy()

    existing_identifier_cols = [
        col for col in CIRA_IDENTIFIER_COLUMNS
        if col in feature_df.columns
    ]

    if existing_identifier_cols:
        feature_df = feature_df.drop(columns=existing_identifier_cols)

    leaked_targets = [
        col for col in CIRA_TARGET_COLUMNS
        if col in feature_df.columns
    ]

    if leaked_targets:
        raise ValueError(
            "Target columns found among CIRA input features: "
            f"{leaked_targets}. Label/DoH must not be part of X."
        )

    leaked_identifiers = [
        col for col in CIRA_IDENTIFIER_COLUMNS
        if col in feature_df.columns
    ]

    if leaked_identifiers:
        raise ValueError(
            "Identifier columns found among CIRA input features: "
            f"{leaked_identifiers}. Remove identifiers before adversarial training."
        )

    feature_cols = list(feature_df.columns)

    if len(feature_cols) != CIRA_EXPECTED_FEATURES:
        raise ValueError(
            f"Expected {CIRA_EXPECTED_FEATURES} CIRA traffic features, "
            f"but received {len(feature_cols)}. Use either the full CIRA dataframe "
            "or the filtered dataframe with exactly 29 traffic features plus Label/DoH."
        )

    X = feature_df.loc[:, feature_cols].apply(pd.to_numeric, errors="coerce")

    valid_rows = X.notna().all(axis=1) & y.notna()
    X = X.loc[valid_rows].copy()
    y = y.loc[valid_rows].astype("int32").copy()

    return X, y, feature_cols


def build_cira_plausibility_mask(feature_cols, mask_mode=CIRA_MASK_MODE):
    """
    Builds a CIRA plausibility mask over the 29 numerical traffic features.

    Mask semantics:
    - 1.0 means the feature may be perturbed.
    - 0.0 means the feature is blocked and must remain unchanged.

    The function uses a deny-by-default strategy:
    - all features are blocked by default;
    - only features from the selected CIRA_ALLOWED_FEATURES_* list are enabled.
    """
    feature_cols = list(feature_cols)

    if len(feature_cols) != CIRA_EXPECTED_FEATURES:
        raise ValueError(
            f"Expected {CIRA_EXPECTED_FEATURES} CIRA input features, "
            f"but received {len(feature_cols)}."
        )

    leaked_targets = [
        col for col in CIRA_TARGET_COLUMNS
        if col in feature_cols
    ]

    if leaked_targets:
        raise ValueError(
            "Target columns found in the feature list: "
            f"{leaked_targets}. Label/DoH must not be part of the mask."
        )

    leaked_identifiers = [
        col for col in CIRA_IDENTIFIER_COLUMNS
        if col in feature_cols
    ]

    if leaked_identifiers:
        raise ValueError(
            "Identifier columns found in the feature list: "
            f"{leaked_identifiers}. Identifiers must not be perturbed."
        )

    allowed_features = get_cira_allowed_features(mask_mode)

    missing_allowed = [
        feature for feature in allowed_features
        if feature not in feature_cols
    ]

    if missing_allowed:
        raise ValueError(
            "The following expected perturbable CIRA features were not found: "
            f"{missing_allowed}"
        )

    mask_series = pd.Series(0.0, index=feature_cols, dtype="float32")
    mask_series.loc[allowed_features] = 1.0

    return mask_series.values.astype("float32"), mask_series


def print_cira_plausibility_mask_report(mask_series, mask_mode=CIRA_MASK_MODE):
    """
    Prints a compact report for the CIRA plausibility mask.
    """
    allowed = mask_series[mask_series == 1.0]
    blocked = mask_series[mask_series == 0.0]

    print("\n" + "-" * 70)
    print(f"[CIRA Plausibility Mask: {mask_mode.upper()}]")
    print(f"Total input features: {len(mask_series)}")
    print(f"Perturbable features: {len(allowed)}")
    print(f"Blocked features: {len(blocked)}")

    print("\nPerturbable features:")
    for name in allowed.index:
        print(f"  - {name}")

    print("\nBlocked features:")
    for name in blocked.index:
        print(f"  - {name}")
    print("-" * 70)


def configure_gpu():
    """
    Configures TensorFlow GPU memory growth.
    """
    gpus = tf.config.list_physical_devices("GPU")

    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass


def set_seed(seed):
    """
    Sets random seeds for reproducibility.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def build_mtl_model_M1(input_shape):
    inputs = layers.Input(shape=input_shape)

    def eca_block(input_tensor):
        channels = input_tensor.shape[-1]
        squeeze = layers.GlobalAveragePooling2D()(input_tensor)
        squeeze = layers.Reshape((1, 1, channels))(squeeze)
        k_size = max(3, int(abs((np.log2(channels) + 1) / 2 + 0.5)))
        squeeze = layers.Conv2D(
            1,
            kernel_size=(1, k_size),
            padding="same",
            activation="sigmoid",
            use_bias=False,
        )(squeeze)
        return layers.Multiply()([input_tensor, squeeze])

    def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.3):
        x = layers.LayerNormalization(epsilon=1e-6)(inputs)
        x = layers.MultiHeadAttention(
            key_dim=head_size,
            num_heads=num_heads,
            dropout=dropout,
        )(x, x)
        x = layers.Dropout(dropout)(x)
        res = x + inputs

        x = layers.LayerNormalization(epsilon=1e-6)(res)
        x = layers.Dense(ff_dim, activation="relu")(x)
        x = layers.Dropout(dropout)(x)
        x = layers.Dense(inputs.shape[-1])(x)

        return x + res

    # Convolutional Block 1
    x = layers.Conv2D(32, (3, 3), padding="same")(inputs)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    # Convolutional Block 2
    x = layers.Conv2D(64, (3, 3), padding="same")(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    # Convolutional Block 3
    x = layers.Conv2D(128, (3, 3), padding="same")(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    x = layers.Reshape((-1, x.shape[-1]))(x)
    x = transformer_encoder(
        x,
        head_size=128,
        num_heads=4,
        ff_dim=256,
        dropout=0.3,
    )
    x = layers.Flatten()(x)

    x = layers.Dense(128, activation="relu", kernel_regularizer=l2(1e-2))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=l2(1e-2))(x)

    logits = layers.Dense(1, activation=None, name="logits")(x)
    output = layers.Activation(
        "sigmoid",
        dtype="float32",
        name="binary_output",
    )(logits)

    model = Model(inputs=inputs, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss=BinaryCrossentropy(),
        metrics=["accuracy", "Precision", "Recall", "AUC"],
    )

    return model


# ------------------------------------------------------------
# METRICS AND THRESHOLD
# ------------------------------------------------------------
def get_best_threshold(y_true, y_pred_proba):
    best_t = 0.5
    best_f1 = 0.0

    y_true_flat = np.asarray(y_true).reshape(-1)
    y_pred_proba_flat = np.asarray(y_pred_proba).reshape(-1)

    for t in np.arange(0.01, 1.0, 0.01):
        y_pred = (y_pred_proba_flat > t).astype("int32")
        current_f1 = f1_score(y_true_flat, y_pred, zero_division=0)

        if current_f1 > best_f1:
            best_f1 = current_f1
            best_t = t

    return best_t


def get_metrics(y_true, y_pred_proba, threshold):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred_proba = np.asarray(y_pred_proba).reshape(-1)
    y_pred = (y_pred_proba > threshold).astype("int32")

    return {
        "Acc": accuracy_score(y_true, y_pred),
        "Prec": precision_score(y_true, y_pred, zero_division=0),
        "Rec": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_true, y_pred_proba),
    }


def calculate_ASR_I(y_true, y_clean_proba, y_adv_proba, threshold):
    y_clean_pred = (
        np.asarray(y_clean_proba).reshape(-1) > threshold
    ).astype("int32")

    y_adv_pred = (
        np.asarray(y_adv_proba).reshape(-1) > threshold
    ).astype("int32")

    originally_malicious = (
        np.asarray(y_true).reshape(-1) == 1
    ) & (y_clean_pred == 1)

    successfully_evaded = originally_malicious & (y_adv_pred == 0)

    denominator = np.sum(originally_malicious)

    return np.sum(successfully_evaded) / denominator if denominator > 0 else 0.0


def reshape_to_2d(data, size):
    pad_size = size**2 - data.shape[1]

    padded = np.pad(
        data,
        pad_width=((0, 0), (0, pad_size)),
        mode="constant",
    )

    return padded.reshape(-1, size, size, 1).astype("float32")


def prepare_splits(X, y, seed):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=seed,
        stratify=y,
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=seed,
        stratify=y_temp,
    )

    smote = SMOTE(random_state=seed)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    scaler = MinMaxScaler()

    X_train_scaled = scaler.fit_transform(X_train_res).astype("float32")
    X_val_scaled = scaler.transform(X_val).astype("float32")
    X_test_scaled = scaler.transform(X_test).astype("float32")

    size = int(np.ceil(np.sqrt(X_train_scaled.shape[1])))

    return (
        reshape_to_2d(X_train_scaled, size),
        reshape_to_2d(X_val_scaled, size),
        reshape_to_2d(X_test_scaled, size),
        np.asarray(y_train_res).astype("float32"),
        np.asarray(y_val).astype("float32"),
        np.asarray(y_test).astype("float32"),
        size,
        scaler,
    )


def make_tf_dataset(X_data, y_data, batch_size, shuffle=False, seed=42):
    X_data = X_data.astype("float32")
    y_data = np.asarray(y_data).astype("float32").reshape(-1, 1)

    with tf.device("/CPU:0"):
        dataset = tf.data.Dataset.from_tensor_slices((X_data, y_data))

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=min(len(X_data), 10000),
            seed=seed,
            reshuffle_each_iteration=True,
        )

    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


# ------------------------------------------------------------
# WHITE-BOX ADVERSARIAL ATTACKS
# ------------------------------------------------------------
bce_loss_fn = tf.keras.losses.BinaryCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE,
)


@tf.function(reduce_retracing=True)
def fgsm_step_fn(logit_model, x_batch_tensor, y_batch_tensor, epsilon, mask_tensor):
    with tf.GradientTape() as tape:
        tape.watch(x_batch_tensor)
        logits = logit_model(x_batch_tensor, training=False)
        loss = bce_loss_fn(y_batch_tensor, logits)

    gradient = tape.gradient(loss, x_batch_tensor)
    masked_gradient = gradient * mask_tensor

    return tf.clip_by_value(
        x_batch_tensor + epsilon * tf.sign(masked_gradient),
        0.0,
        1.0,
    )


def fgsm_attack_batched(logit_model, x, y, epsilon, feature_mask_2d, batch_size=256):
    """
    Generates masked FGSM adversarial examples.

    The mask is applied directly to the input gradient so blocked CIRA features
    remain unchanged.
    """
    x_adv_list = []
    y_array = np.asarray(y).reshape(-1).astype("float32")
    mask_tensor = tf.convert_to_tensor(feature_mask_2d, dtype=tf.float32)

    for start in range(0, len(x), batch_size):
        end = start + batch_size

        x_tensor = tf.convert_to_tensor(
            x[start:end].astype("float32"),
            dtype=tf.float32,
        )

        y_tensor = tf.convert_to_tensor(
            y_array[start:end].reshape(-1, 1),
            dtype=tf.float32,
        )

        x_adv_list.append(
            fgsm_step_fn(
                logit_model,
                x_tensor,
                y_tensor,
                epsilon,
                mask_tensor,
            ).numpy()
        )

        gc.collect()

    return np.concatenate(x_adv_list, axis=0).astype("float32")


@tf.function(reduce_retracing=True)
def pgd_step_fn(logit_model, x_adv_tensor, x_orig_tensor, y_tensor, epsilon, alpha, mask_tensor):
    with tf.GradientTape() as tape:
        tape.watch(x_adv_tensor)
        logits = logit_model(x_adv_tensor, training=False)
        loss = bce_loss_fn(y_tensor, logits)

    gradient = tape.gradient(loss, x_adv_tensor)
    masked_gradient = gradient * mask_tensor

    x_adv_tensor = x_adv_tensor + alpha * tf.sign(masked_gradient)

    perturbation = tf.clip_by_value(
        x_adv_tensor - x_orig_tensor,
        -epsilon,
        epsilon,
    )

    # Apply the mask again for numerical safety after projection.
    perturbation = perturbation * mask_tensor

    return tf.clip_by_value(x_orig_tensor + perturbation, 0.0, 1.0)


def pgd_attack_batched(logit_model, x, y, epsilon, alpha, steps, feature_mask_2d, batch_size=256):
    """
    Generates masked PGD adversarial examples.

    The mask is applied at every PGD step so blocked CIRA features remain fixed.
    """
    x_adv_list = []
    y_array = np.asarray(y).reshape(-1).astype("float32")
    mask_tensor = tf.convert_to_tensor(feature_mask_2d, dtype=tf.float32)

    for start in range(0, len(x), batch_size):
        end = start + batch_size

        x_original = tf.convert_to_tensor(
            x[start:end].astype("float32"),
            dtype=tf.float32,
        )

        x_adv = tf.identity(x_original)

        y_tensor = tf.convert_to_tensor(
            y_array[start:end].reshape(-1, 1),
            dtype=tf.float32,
        )

        for _ in range(steps):
            x_adv = pgd_step_fn(
                logit_model,
                x_adv,
                x_original,
                y_tensor,
                epsilon,
                alpha,
                mask_tensor,
            )

        x_adv_list.append(x_adv.numpy())
        gc.collect()

    return np.concatenate(x_adv_list, axis=0).astype("float32")


def evaluate_adversarial(
    model,
    X_test_2d,
    y_test,
    epsilons,
    y_clean_proba,
    best_threshold,
    feature_mask_2d,
):
    results = []
    logit_model = tf.keras.Model(
        inputs=model.input,
        outputs=model.get_layer("logits").output,
    )

    total_fgsm_gen_time = 0.0
    total_fgsm_inf_time = 0.0
    total_pgd_gen_time = 0.0
    total_pgd_inf_time = 0.0

    for eps in epsilons:
        # FGSM generation
        t0_gen = time.perf_counter()
        X_fgsm = fgsm_attack_batched(
            logit_model,
            X_test_2d,
            y_test,
            eps,
            feature_mask_2d,
            ADV_BATCH_SIZE,
        )
        t_gen_fgsm = time.perf_counter() - t0_gen
        total_fgsm_gen_time += t_gen_fgsm

        # PGD generation
        t0_gen = time.perf_counter()
        X_pgd = pgd_attack_batched(
            logit_model,
            X_test_2d,
            y_test,
            eps,
            eps / 4,
            10,
            feature_mask_2d,
            ADV_BATCH_SIZE,
        )
        t_gen_pgd = time.perf_counter() - t0_gen
        total_pgd_gen_time += t_gen_pgd

        # FGSM inference
        t0_inf = time.perf_counter()
        y_fgsm_proba = model.predict(
            X_fgsm,
            batch_size=ADV_BATCH_SIZE,
            verbose=0,
        )
        t_inf_fgsm = time.perf_counter() - t0_inf
        total_fgsm_inf_time += t_inf_fgsm

        # PGD inference
        t0_inf = time.perf_counter()
        y_pgd_proba = model.predict(
            X_pgd,
            batch_size=ADV_BATCH_SIZE,
            verbose=0,
        )
        t_inf_pgd = time.perf_counter() - t0_inf
        total_pgd_inf_time += t_inf_pgd

        fgsm_metrics = get_metrics(y_test, y_fgsm_proba, best_threshold)
        fgsm_metrics["ASR_I"] = calculate_ASR_I(
            y_test,
            y_clean_proba,
            y_fgsm_proba,
            best_threshold,
        )
        fgsm_metrics["Gen_Time"] = t_gen_fgsm
        fgsm_metrics["Inf_Time"] = t_inf_fgsm

        pgd_metrics = get_metrics(y_test, y_pgd_proba, best_threshold)
        pgd_metrics["ASR_I"] = calculate_ASR_I(
            y_test,
            y_clean_proba,
            y_pgd_proba,
            best_threshold,
        )
        pgd_metrics["Gen_Time"] = t_gen_pgd
        pgd_metrics["Inf_Time"] = t_inf_pgd

        results.append(
            {
                "epsilon": eps,
                "FGSM": fgsm_metrics,
                "PGD": pgd_metrics,
            }
        )

        del X_fgsm, X_pgd, y_fgsm_proba, y_pgd_proba
        gc.collect()

    return (
        results,
        total_fgsm_gen_time,
        total_fgsm_inf_time,
        total_pgd_gen_time,
        total_pgd_inf_time,
    )


def run_single_experiment(X, y, feature_mask_1d, run_id, seed):
    tf.keras.backend.clear_session()
    gc.collect()
    set_seed(seed)

    (
        X_train_2d,
        X_val_2d,
        X_test_2d,
        y_train_res,
        y_val,
        y_test,
        size,
        scaler,
    ) = prepare_splits(X, y, seed)

    # Reshape the 1D feature mask to the same padded 2D layout used by Conv2D.
    feature_mask_2d = reshape_to_2d(
        np.array([feature_mask_1d], dtype="float32"),
        size,
    )[0:1]

    train_ds = make_tf_dataset(
        X_train_2d,
        y_train_res,
        BATCH_SIZE,
        shuffle=True,
        seed=seed,
    )

    val_ds = make_tf_dataset(
        X_val_2d,
        y_val,
        BATCH_SIZE,
        shuffle=False,
        seed=seed,
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
        ),
    ]

    model = build_mtl_model_M1(input_shape=(size, size, 1))

    start_train = time.perf_counter()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    train_time = time.perf_counter() - start_train
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

    y_val_proba = model.predict(
        X_val_2d.astype("float32"),
        batch_size=ADV_BATCH_SIZE,
        verbose=0,
    )

    best_threshold = get_best_threshold(y_val, y_val_proba)

    y_clean_proba = model.predict(
        X_test_2d.astype("float32"),
        batch_size=ADV_BATCH_SIZE,
        verbose=0,
    )

    clean_metrics = get_metrics(y_test, y_clean_proba, best_threshold)

    y_clean_pred = (y_clean_proba.reshape(-1) > best_threshold).astype("int32")
    y_test_flat = np.asarray(y_test).reshape(-1).astype("int32")
    originally_malicious = (y_test_flat == 1) & (y_clean_pred == 1)

    print("\n" + "-" * 70)
    print(f"[Run Diagnostic {run_id}]")
    print(f"Best Threshold (Validation): {best_threshold:.4f}")
    print("Clean metrics:", clean_metrics)
    print("y_test distribution:", np.bincount(y_test_flat))
    print("Clean prediction distribution:", np.bincount(y_clean_pred))
    print("Correctly detected malicious samples:", int(originally_malicious.sum()))
    print(
        "Clean probabilities - min/max/mean:",
        float(y_clean_proba.min()),
        float(y_clean_proba.max()),
        float(y_clean_proba.mean()),
    )
    print("-" * 70)

    adv_results, t_fgsm_gen, t_fgsm_inf, t_pgd_gen, t_pgd_inf = evaluate_adversarial(
        model,
        X_test_2d,
        y_test,
        EPSILONS,
        y_clean_proba,
        best_threshold,
        feature_mask_2d,
    )

    print(f"\n[Run {run_id} Completed]")
    print(f"-> Training Time: {train_time:.2f}s")
    print(
        "-> Accumulated FGSM Time "
        f"(Generation: {t_fgsm_gen:.2f}s | "
        f"Inference: {t_fgsm_inf:.2f}s | "
        f"Total: {t_fgsm_gen + t_fgsm_inf:.2f}s)"
    )
    print(
        "-> Accumulated PGD Time  "
        f"(Generation: {t_pgd_gen:.2f}s | "
        f"Inference: {t_pgd_inf:.2f}s | "
        f"Total: {t_pgd_gen + t_pgd_inf:.2f}s)"
    )

    for adv in adv_results:
        print(f"   -> Eps={adv['epsilon']}")
        print(
            f"      [FGSM] F1={adv['FGSM']['F1']:.4f}, "
            f"ASR_I={adv['FGSM']['ASR_I']:.4f} | "
            f"Gen Time={adv['FGSM']['Gen_Time']:.4f}s, "
            f"Inf={adv['FGSM']['Inf_Time']:.4f}s"
        )
        print(
            f"      [PGD]  F1={adv['PGD']['F1']:.4f}, "
            f"ASR_I={adv['PGD']['ASR_I']:.4f} | "
            f"Gen Time={adv['PGD']['Gen_Time']:.4f}s, "
            f"Inf={adv['PGD']['Inf_Time']:.4f}s"
        )

    result = {
        "run": run_id,
        "seed": seed,
        "best_epoch": best_epoch,
        "train_time": train_time,
        "best_threshold": best_threshold,
        "clean": {
            **clean_metrics,
            "n_detected_malicious_clean": int(originally_malicious.sum()),
        },
        "adversarial": adv_results,
        "total_fgsm_gen": t_fgsm_gen,
        "total_fgsm_inf": t_fgsm_inf,
        "total_pgd_gen": t_pgd_gen,
        "total_pgd_inf": t_pgd_inf,
    }

    del model, train_ds, val_ds
    del X_train_2d, X_val_2d, X_test_2d
    del y_train_res, y_val, y_test, y_clean_proba

    gc.collect()
    tf.keras.backend.clear_session()

    return result


def _worker_run(run_id, seed, X, y, feature_mask_1d, return_dict):
    configure_gpu()

    result = run_single_experiment(
        X,
        y,
        feature_mask_1d,
        run_id,
        seed,
    )

    return_dict[run_id] = result


# ------------------------------------------------------------
# RESULT PROCESSING
# ------------------------------------------------------------
def summarize_adversarial_results(all_results):
    print("\n" + "=" * 80)
    print("ADVERSARIAL RESULTS (WHITE-BOX CIRA): MEAN ± STANDARD DEVIATION")
    print("=" * 80)

    thresholds = [r.get("best_threshold", 0.5) for r in all_results]

    print(
        f"\n[Info] Mean Optimal Threshold: "
        f"{np.mean(thresholds):.4f} ± {np.std(thresholds):.4f}"
    )

    fgsm_gen_totals = [r["total_fgsm_gen"] for r in all_results]
    fgsm_inf_totals = [r["total_fgsm_inf"] for r in all_results]
    pgd_gen_totals = [r["total_pgd_gen"] for r in all_results]
    pgd_inf_totals = [r["total_pgd_inf"] for r in all_results]

    print("\n[ACCUMULATED TIMES PER RUN (All Epsilons)]")
    print(
        f"FGSM - Total Generation: "
        f"{np.mean(fgsm_gen_totals):.4f}s ± {np.std(fgsm_gen_totals):.4f}s"
    )
    print(
        f"FGSM - Total Inference:  "
        f"{np.mean(fgsm_inf_totals):.4f}s ± {np.std(fgsm_inf_totals):.4f}s"
    )
    print(
        f"PGD  - Total Generation: "
        f"{np.mean(pgd_gen_totals):.4f}s ± {np.std(pgd_gen_totals):.4f}s"
    )
    print(
        f"PGD  - Total Inference:  "
        f"{np.mean(pgd_inf_totals):.4f}s ± {np.std(pgd_inf_totals):.4f}s"
    )

    for attack in ["FGSM", "PGD"]:
        print("\n" + "-" * 40)
        print(f"Attack: {attack} (Metrics by Epsilon)")
        print("-" * 40)

        for eps in EPSILONS:
            print(f"Epsilon = {eps}")

            for metric in ["Acc", "Prec", "Rec", "F1", "AUC", "ASR_I", "Gen_Time", "Inf_Time"]:
                values = [
                    adv_result[attack][metric]
                    for r in all_results
                    for adv_result in r["adversarial"]
                    if adv_result["epsilon"] == eps
                ]

                unit = "s" if "Time" in metric else ""

                print(
                    f"  {metric}: "
                    f"{np.mean(values):.4f}{unit} ± {np.std(values):.4f}{unit}"
                )


def export_results_to_csv(
    all_results,
    output_path="adversarial_metrics_whitebox_CIRA_mask_operational.csv",
):
    rows = []

    for result in all_results:
        run = result["run"]
        seed = result["seed"]
        best_t = result.get("best_threshold", 0.5)
        clean = result.get("clean", {})

        for adv in result["adversarial"]:
            eps = adv["epsilon"]

            for attack in ["FGSM", "PGD"]:
                row = {
                    "run": run,
                    "seed": seed,
                    "condition": "adversarial",
                    "scenario": f"whitebox_cira_mask_{CIRA_MASK_MODE}",
                    "attack": attack,
                    "epsilon": eps,
                    "threshold": best_t,
                    **adv[attack],
                    "clean_Acc": clean.get("Acc", np.nan),
                    "clean_Prec": clean.get("Prec", np.nan),
                    "clean_Rec": clean.get("Rec", np.nan),
                    "clean_F1": clean.get("F1", np.nan),
                    "clean_AUC": clean.get("AUC", np.nan),
                    "n_detected_malicious_clean": clean.get(
                        "n_detected_malicious_clean",
                        np.nan,
                    ),
                    "best_epoch": result["best_epoch"],
                    "train_time": result["train_time"],
                    "total_run_gen_time": result[f"total_{attack.lower()}_gen"],
                    "total_run_inf_time": result[f"total_{attack.lower()}_inf"],
                    "cira_mask_mode": CIRA_MASK_MODE,
                }

                rows.append(row)

    df_results = pd.DataFrame(rows)
    df_results.to_csv(output_path, index=False)

    print(f"\nResults saved to: {output_path}")

    return df_results


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main(df_filtrado, feature_mask_1d=None, mask_mode=CIRA_MASK_MODE):
    """
    Executes N_RUNS runs of white-box adversarial evaluation with a CIRA
    plausibility mask.

    Accepted inputs:
    - full CIRA dataframe containing identifier columns and Label/DoH;
    - filtered CIRA dataframe containing the 29 traffic features and Label/DoH.

    Label/DoH is a response variable and is never used as an input feature or as
    part of the plausibility mask.

    Parameters
    ----------
    df_filtrado : pandas.DataFrame
        CIRA dataframe.
    feature_mask_1d : None, array-like, or pandas.Series
        Optional external binary mask. If None, a mask is built from feature
        names using mask_mode.
    mask_mode : str
        One of {"strict", "operational", "extended"} when feature_mask_1d is
        None. Default is "operational".
    """
    try:
        mp.set_start_method("spawn")
    except RuntimeError:
        pass

    df = df_filtrado.copy()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)

    X, y, feature_cols = select_cira_features_and_target(df)

    if feature_mask_1d is None:
        print("\n[Info] No external mask received.")
        print(f"[Info] Building the CIRA {mask_mode!r} plausibility mask from feature names.")
        feature_mask_1d, mask_series = build_cira_plausibility_mask(
            feature_cols,
            mask_mode=mask_mode,
        )
    else:
        if isinstance(feature_mask_1d, pd.Series):
            feature_mask_1d = feature_mask_1d.reindex(feature_cols).values

        feature_mask_1d = np.asarray(feature_mask_1d, dtype="float32").reshape(-1)

        if len(feature_mask_1d) != X.shape[1]:
            raise ValueError(
                f"feature_mask_1d has length {len(feature_mask_1d)}, "
                f"but X has {X.shape[1]} features."
            )

        if np.isnan(feature_mask_1d).any():
            raise ValueError(
                "feature_mask_1d contains NaN values. If a pandas Series was "
                "provided, make sure its index matches the CIRA feature names."
            )

        invalid_values = set(np.unique(feature_mask_1d)) - {0.0, 1.0}
        if invalid_values:
            raise ValueError(
                "feature_mask_1d must be binary, with values 0.0 or 1.0. "
                f"Found invalid values: {sorted(invalid_values)}"
            )

        mask_series = pd.Series(feature_mask_1d, index=feature_cols, dtype="float32")

    if len(feature_mask_1d) != X.shape[1]:
        raise ValueError(
            f"feature_mask_1d has length {len(feature_mask_1d)}, "
            f"but X has {X.shape[1]} features."
        )

    print_cira_plausibility_mask_report(mask_series, mask_mode=mask_mode)

    all_results = []
    manager = mp.Manager()
    return_dict = manager.dict()

    print("\n" + "=" * 50)
    print(f"STARTING {N_RUNS} RUNS (WHITE-BOX CIRA MASK: {str(mask_mode).upper()})")
    print("=" * 50)

    for run in range(N_RUNS):
        seed = 42 + run

        print(
            f"\n[{time.strftime('%H:%M:%S')}] "
            f"Isolated Process -> Run {run + 1}/{N_RUNS}"
        )

        p = mp.Process(
            target=_worker_run,
            args=(run + 1, seed, X, y, feature_mask_1d, return_dict),
        )

        p.start()
        p.join()

        if (run + 1) in return_dict:
            all_results.append(return_dict[run + 1])
            del return_dict[run + 1]

    summarize_adversarial_results(all_results)
    df_results = export_results_to_csv(all_results)

    return all_results, df_results


In [ ]:
import importlib
import ids_engine

importlib.reload(ids_engine)

In [ ]:
all_results, df_results = ids_engine.main(df_filtrado)